In [3]:
version = "v2.2.3"

In [4]:
# update this string!
student_name = "Daniel_"

In [5]:
print(f"Notebook version {version}")

if student_name == "First Last":
    raise Exception(
        "Please update your name in the 'student_name' variable at the top of this notebook."
    )

print(f"Student name: {student_name}")

Notebook version v2.2.3
Student name: Daniel_


# **Course Minimum Score Notice**
- Student must achieve a _**Passing**_ score on **ALL** course assignments and quizzes to receive credit for this course.  
- What defines a _**Passing**_ score?
  - Students must attain a **Raw Score** of not less than 80.0% on **All** assignments and quizzes.  
  - The **Raw Score** is that score achieved prior to the _final_ application of incurred late penalties. 

<a id='toc'></a>
# Table of Contents
- **[Assignment 3 Description](#Topic0)**
  - [Task 1 - Spam dataset import](#t1)
  - [Data Preparation](#dataprep)
  - [Task 2 - Dummy Classifiers](#t2)
  - [Task 3 - SVC Classifier](#t3)
  - [Task 4 - SVC and the Decision Function](#t4)
    - [SVC Confusion Matrix Vizualization](#svmcmviz)
  - [Task 5 - Logistic Regression Spam Classifier](#t5)
    - [Precision-Recall Curve Vizualization](#prcurveviz)
  - [Task 6 - Logistic Regression Confusion Matrix](#t6)
    - [Logistic Regression Confusion Matrix Visualization](#lrcmviz)
  - [Task 7 - Confusion Matrix Metrics](#t7)
  - [Task 8 - Grid Search on Logistic Regression](#t8)
    - [GridSearch Heatmap Visualization](#heatmapviz)
  - [Task 9 - Normalizing features and Regularization](#t9)

In [7]:
# Either of the following is no longer
# necessary for matplotlib in notebooks.
# The import statement has you covered!

# %matplotlib notebook
# %matplotlib inline

<a id='Topic0'></a>
# Assignment 3: Classification and Evaluation

In this assignment we will build several classification models and calculate some useful metrics for gauging the performance of these models. The scenario we'll address is "spam" e-mail detection: a very important, widely-used supervised machine learning task that attempts to find unsolicited, mass-produced messages that have irrelevant and/or inappropriate content (often mass marketing or attempts at fraud). These are sometimes called "spam filters".

We treat this task as a binary classification problem: detecting if an email is "spam" (Class == 1) or not (Class == 0, a regular/good e-mail). Email systems will typically automatically move messages detected as "spam" to a "Spam" or "Deleted" folder so the user will not have to read them in their regular inbox.

In this setup, a *false positive* would mark a regular/good e-mail as spam. The key aspect of the "spam" scenario is that false positives are obviously very undesirable, because these would cause people to potentially lose valuable "good" messages. So we want a highly precise spam filter that has few/no false positives, but as we'll see, as a consequence it may let more spam through the filter. This is a classic precision / recall tradeoff, which you'll investigate below.  

<a href='#toc'>TOC</a>

In [8]:
# Suppress all warnings only when absolutely necessary
# Warnings are in place for a reason!
import warnings

# warnings.filterwarnings('ignore')
# warnings.simplefilter('ignore')

In [9]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from mads.lib.path import assets

up, down = True, False
labels = {"ham (neg)": 0, "spam (pos)": 1}.keys()

In [10]:
## Additional imports can be inlcuded here

np.set_printoptions(precision=5)

### Confusion Matrix simple plot function

In [11]:
def plot_confusion(cm, labels):
    display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    display_cm.plot()
    plt.show()

<a id='t1'></a>
### Task 1 - Spam dataset import and class label ratio (5 points).

Import the data from `spam.csv` using `assets.find("spam.csv")`. What is the ratio of the counts of regular (class == 0) to spam (Class == 1) observations in the entire dataset? 

*This function should return a positive float less than 10.*  

<a href='#toc'>TOC</a>

In [12]:
# hidden autograder codeblock
task_id = "1"

In [22]:
import pandas as pd
from mads.lib.path import assets

def task_one():
    r = None

    # YOUR CODE HERE

    df = pd.read_csv(assets.find("spam.csv"))
    
    # 2. Identify the label column
    # Usually, the labels are in the last column. 
    # This line identifies the name of the last column automatically.
    label_col = df.columns[-1] 
    
    # 3. Count the occurrences
    counts = df[label_col].value_counts()
    
    # 4. Calculate the ratio (Class 0 / Class 1)
    # counts[0] = regular, counts[1] = spam
    r = float(counts[0] / counts[1])
    #raise NotImplementedError()

    return r

In [23]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

task_one()

1.5377826806398236

In [24]:
# Autograder tests
print(f"Task {task_id} - AG tests")

stu_ans = task_one()

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(
    stu_ans, float
), f"Task {task_id}: Your function should return a float. "

assert 0.0 <= stu_ans <= 10.0, f"Task {task_id}: Your answer must be between 0 and 10. "

# Some hidden tests

del stu_ans

Task 1 - AG tests
Task 1 - your answer:
1.5377826806398236


<a id='dataprep'></a>
### Data Preparation
We break into training and testing sets as usual. But there's another critical step, since we're going to be using *regularized* classification methods on this data. We must first *perform feature normalization so that all features are on a standardized scale.*  We do this by applying the StandardScalar class from sklearn.preprocessing.

The details of how we do this are important. You must *first* split the data into training and test sets, and *only after the split*, do the feature normalization. This is to avoid giving information about the range of variables in the test split to the training split, which would be a form of data leakage.  

**Note:** Use X_train, X_test, y_train, y_test for all assignment tasks.  
Also, X_train_raw and X_test_raw will be useful for Question 7.  

<a href='#toc'>TOC</a>

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

file = assets.find("spam.csv")
df = pd.read_csv(file)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, random_state=0)

scaler = StandardScaler().fit(X_train_raw)

X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

<a id='t2'></a>
### Task 2 - Dummy Classifiers (15 points).

We've seen that the so-called [DummyClassifiers](https://scikit-learn.org/1.2/modules/generated/sklearn.dummy.DummyClassifier.html#sklearn.dummy.DummyClassifier) can be used as a simple *sanity-check* baseline against which to compare real classifier performance. If your classifier can't do much better than the dummy classifier, you probably have more work to do.

Using `X_train`, `X_test`, `y_train`, and `y_test`, train two dummy classifiers: (A) one that respects the training set's label distribution and (B) one that classifies everything as the majority class of the training data. Where appropriate, make sure to set the *random_state* parameter to zero. 

Then on the test set, for each of the classifiers A and B, compute precision, recall, and accuracy. Report your results as a single tuple as shown below. Once you have the results, it's instructive to compare these two different types of dummy baselines to understand why they are different.  

[precision_score](https://scikit-learn.org/1.2/modules/generated/sklearn.metrics.precision_score.html#sklearn.metrics.precision_score)  
[recall_score](https://scikit-learn.org/1.2/modules/generated/sklearn.metrics.recall_score.html#sklearn.metrics.recall_score)  

*This function should a return a tuple of six floats, like so:*

*`(precision_score_A, recall_score_A, accuracy_score_A, precision_score_B, recall_score_B, accuracy_score_B)`.*  

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "2"

In [26]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score

def task_two():
    preA, recA, accA, preB, recB, accB = None, None, None, None, None, None

    # YOUR CODE HERE
    # --- Classifier A: Stratified (Respects label distribution) ---
    # We use random_state=0 as requested in the instructions
    dummy_a = DummyClassifier(strategy='stratified', random_state=0)
    dummy_a.fit(X_train, y_train)
    y_pred_a = dummy_a.predict(X_test)
    
    # Calculate metrics for A
    preA = float(precision_score(y_test, y_pred_a))
    recA = float(recall_score(y_test, y_pred_a))
    accA = float(accuracy_score(y_test, y_pred_a))
    
    # --- Classifier B: Most Frequent (Classifies as majority class) ---
    dummy_b = DummyClassifier(strategy='most_frequent', random_state=0)
    dummy_b.fit(X_train, y_train)
    y_pred_b = dummy_b.predict(X_test)
    
    # Calculate metrics for B
    preB = float(precision_score(y_test, y_pred_b, zero_division=0))
    recB = float(recall_score(y_test, y_pred_b))
    accB = float(accuracy_score(y_test, y_pred_b))
    
    #raise NotImplementedError()

    return preA, recA, accA, preB, recB, accB

In [28]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

task_two()

(0.3995433789954338,
 0.3804347826086957,
 0.523892267593397,
 0.0,
 0.0,
 0.6003475238922676)

In [29]:
# Autograder tests
print(f"Task {task_id} - AG tests")
stu_ans = task_two()
print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(
    stu_ans, tuple
), f"Task {task_id}: Your function should return a tuple. "

assert (
    len(stu_ans) == 6
), f"Task {task_id}: The length of your returned tuple should be 6. "

assert all(
    [isinstance(item, float) for item in stu_ans]
), f"Task {task_id}: Your tuple should only contain floats. "

# Some hidden tests

del stu_ans

Task 1 - AG tests
Task 1 - your answer:
(0.3995433789954338, 0.3804347826086957, 0.523892267593397, 0.0, 0.0, 0.6003475238922676)


<a id='t3'></a>
### Task 3 - SVC Classifier (15 points).

Using `X_train`, `X_test`, `y_train`, and `y_test`, train an [SVC Classifier](https://scikit-learn.org/1.2/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC) with the default hyper-parameters. What are the accuracy, recall and precision of this classifier on the testing set?

*This function should a return a tuple of three floats, i.e. `(accuracy score, recall score, precision score)`.*  

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "3"

In [30]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score

def task_three():
    acc, rec, pre = (None,) * 3

    # YOUR CODE HERE

    # Initialize variables
    acc, rec, pre = (None,) * 3
    
    # 1. Instantiate the SVC classifier with default hyper-parameters
    svc_model = SVC()
    
    # 2. Train the model using the training data
    svc_model.fit(X_train, y_train)
    
    # 3. Predict on the test set
    y_pred = svc_model.predict(X_test)
    
    # 4. Compute metrics
    # Note: Order must be (accuracy, recall, precision) as per instructions
    acc = float(accuracy_score(y_test, y_pred))
    rec = float(recall_score(y_test, y_pred))
    pre = float(precision_score(y_test, y_pred))
    #raise NotImplementedError()

    return acc, rec, pre

In [32]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

task_three()

(0.9218071242397915, 0.8673913043478261, 0.9322429906542056)

In [33]:
# Autograder tests
print(f"Task {task_id} - AG tests")
stu_ans = task_three()

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(
    stu_ans, tuple
), f"Task {task_id}: Your function should return a tuple. "

assert (
    len(stu_ans) == 3
), f"Task {task_id}: The length of your returned tuple should be 3. "

assert all(
    [isinstance(item, float) for item in stu_ans]
), f"Task {task_id}: Your tuple should only contain floats. "


# Some hidden tests

del stu_ans

Task 1 - AG tests
Task 1 - your answer:
(0.9218071242397915, 0.8673913043478261, 0.9322429906542056)


<a id='t4'></a>
### Task 4 - SVC and the Decision Function (15 points).

Train an [SVC Classifier](https://scikit-learn.org/1.2/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC) with default hyper-parameters except for `{"C": 1e9, "gamma": 1e-8}`. What is the [Confusion Matrix](https://scikit-learn.org/1.2/modules/generated/sklearn.metrics.confusion_matrix.html#sklearn.metrics.confusion_matrix) on the testing set if we use a threshold of `-100` for the decision function? That is, we classify instances with a raw score greater than -100 under the decision function as Class 1. 

*This function should return a 2x2 numpy array of 4 integers.*  

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "4"

In [34]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

def task_four():
    conf_mtrx = None

    # YOUR CODE HERE
    # 1. Train the SVC with the specific hyperparameters provided
    svc_model = SVC(C=1e9, gamma=1e-8)
    svc_model.fit(X_train, y_train)
    
    # 2. Get the raw decision scores for the test set
    # These are not labels (0/1), but distance-based scores
    scores = svc_model.decision_function(X_test)
    
    # 3. Apply the custom threshold of -100
    # True (1) if score > -100, else False (0)
    y_pred_custom = scores > -100
    
    # 4. Generate the confusion matrix
    # Note: ensure it is a numpy array (which confusion_matrix returns by default)
    conf_mtrx = confusion_matrix(y_test, y_pred_custom)
    #raise NotImplementedError()

    return conf_mtrx

In [35]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# task_four()

In [36]:
# Autograder tests
print(f"Task {task_id} - AG tests")
stu_ans = task_four()

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(
    stu_ans, np.ndarray
), f"Task {task_id}: Your function should return a np.ndarray. "

assert stu_ans.shape == (
    2,
    2,
), f"Task {task_id}: Your confusion matrix should be of size 2x2. "

# Some hidden tests

del stu_ans

Task 1 - AG tests
Task 1 - your answer:
[[653  38]
 [ 89 371]]


<a id='svmcmviz'></a>
### SVC Confusion Matrix Visualization
<a href='#toc'>TOC</a>

In [ ]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# plot_confusion(task_four(), labels)

<a id='t5'></a>
### Task 5 - Logistic Regression Spam Classifier (15 points).

Train a [Logistic Regression](https://scikit-learn.org/1.2/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression) spam e-mail classifier with default hyper-parameters using `X_train` and `y_train`. Create a [Precision-Recall Curve](https://scikit-learn.org/1.2/modules/generated/sklearn.metrics.precision_recall_curve.html#sklearn.metrics.precision_recall_curve) and a Receiver Operating Characteristic [ROC Curve](https://scikit-learn.org/1.2/modules/generated/sklearn.metrics.roc_curve.html#sklearn.metrics.roc_curve) using `y_test` and the probability estimates of being "spam" for X_test.

- Based on the precision-recall curve, what is the recall when the precision is $0.90$?

- Based on the ROC curve, what is the true positive rate when the false positive rate is $0.10$?

Write a function to return the answers. Note you can use a pure programming approach to get the answers: you don't have to actually "plot" the curves. However, it's quite instructive to plot these curves to understand just how e.g. precision and recall trade off. So you can plot the curves and read off the answers. Answers correct up to $\pm 0.02$ are accepted. 

*This function should return a tuple with two floats, i.e. `(recall, true positive rate)`.*  

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "5"

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, roc_curve
import numpy as np

def task_five():
    # Recall and True Positive rate
    rec, tpr = None, None

    # YOUR CODE HERE
    # 1. Train Logistic Regression
    lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    
    # 2. Get probability estimates for the positive class (column 1)
    y_scores = lr.predict_proba(X_test)[:, 1]
    
    # 3. Precision-Recall Curve
    precision, recall, thresholds_pr = precision_recall_curve(y_test, y_scores)
    # Find recall where precision == 0.90
    # We look for the index where precision is closest to 0.90
    idx_pr = np.where(precision >= 0.90)[0][0]
    rec = float(recall[idx_pr])
    
    # 4. ROC Curve
    fpr, tpr_array, thresholds_roc = roc_curve(y_test, y_scores)
    # Find TPR where FPR == 0.10
    # We look for the index where FPR is closest to 0.10
    idx_roc = np.where(fpr >= 0.10)[0][0]
    tpr = float(tpr_array[idx_roc])
    #raise NotImplementedError()

    return rec, tpr

In [38]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# task_five()

In [39]:
# Autograder tests
print(f"Task {task_id} - AG tests")
stu_ans = task_five()

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(
    stu_ans, tuple
), f"Task {task_id}: Your function should return a tuple. "

assert len(stu_ans) == 2, f"Task {task_id}: The length of your tuple should be 2. "

assert (
    stu_ans[0] >= 0.7
), f"Task {task_id}: Your recall value should be greater than 0.7. "

assert (
    stu_ans[1] >= 0.7
), f"Task {task_id}: Your TP rate value should be greater than 0.7. "

# Some hidden tests

del stu_ans

Task 1 - AG tests
Task 1 - your answer:
(0.8586956521739131, 0.908695652173913)


<a id='prcurveviz'></a>
### Precision-Recall Curve Visualization

<a href='#toc'>TOC</a>

In [40]:
from sklearn.metrics import PrecisionRecallDisplay, precision_recall_curve


def plot_PR_curve():
    log_reg = LogisticRegression().fit(X_train, y_train)

    preds = log_reg.predict(X_test)

    pre, rec, _ = precision_recall_curve(y_test, preds)

    fig, ax = plt.subplots(figsize=(8, 6))
    disp = PrecisionRecallDisplay.from_estimator(log_reg, X_test, y_test, ax=ax)

    return

In [ ]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# plot_PR_curve()

<a id='t6'></a>
## Task 6 - Logistic Regression Confusion Matrix (5 Points).

- We now want to better understand the relationship metrics of correct and incorrect predictons made by a binary classification model. A very useful tool for examining prediction performance of binary outcomes, such as we have with our spam dataset, is the [Confusion Matrix](https://scikit-learn.org/1.2/modules/generated/sklearn.metrics.confusion_matrix.html#sklearn.metrics.confusion_matrix)
- Using the SciKit-Learn [LogisticRegression](https://scikit-learn.org/1.2/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression) model, create a function that returns a Confusion Matrix (similar to that produced in Task 4). For this task, your Confusion Matrix can be based on predict() method rather than than the decision_function() as we are not altering the decision threshold.
- Your function definition should include the following parameter.  
  - `random_state: int = 42`
  - Initialize the LogisticRegression() estimator with the following hyperparameters:
    - `max_iter=1000`
    - `random_state=` (use function parameter)
    - Note that all other hyperparameters should retain their default value.
  - Your function should return a confusion matrix in the form of a np.ndarray.  
- Use X_data and y_data provided in the Data Preparation cell above.  
- The output of this Task's function will be used to answer questions in Task 7. Be sure to uncomment to accompanying plot_confusion() function.  
<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "6"

In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

def task_six(random_state: int = 42):
    cm = None

    # YOUR CODE HERE

    # 1. Initialize the LogisticRegression model with specified parameters
    # We use the random_state passed into the function
    lr = LogisticRegression(max_iter=1000, random_state=random_state)
    
    # 2. Fit the model using the training data
    lr.fit(X_train, y_train)
    
    # 3. Generate predictions on the test set using the default threshold
    y_pred = lr.predict(X_test)
    
    # 4. Create the confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    #raise NotImplementedError()

    return cm

In [42]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# task_six(42)

In [43]:
# Autograder tests
print(f"Task {task_id} - AG tests")
stu_ans = task_six()

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(
    stu_ans, np.ndarray
), f"Task {task_id}: The second tuple element should be an np.ndarray"


del stu_ans

Task 1 - AG tests
Task 1 - your answer:
[[648  43]
 [ 68 392]]


<a id='lrcmviz'></a>
### Logistic Regression Confusion Matrix Visualization
<a href='#toc'>TOC</a>

In [ ]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# plot_confusion(task_six(42), labels)

<a id='t7'></a>
## Task 7 - Confusion Matrix Metrics (10 Points).

- Answer the following questions concerning the above Confusion Matrix.  
- Record your answers in the variable names a1,...,a6 in the answer cell below.  
  - You may manually calculate and hardcode your answers or, you can programmatically calculate the answers.
- Your answers should be of Python type **int** or **float**.
 
Q1. What is the number of __Correctly Predicted__ spam emails?  
Q2. How many __False Positive__ predictions (Type I error) were made for spam email?  
Q3. How many __False Negative__ predictions (Type II error) were made for spam email?  
Q4. What is the __Precision__ Score for this spam predictor? (round to three decimal places)  
Q5. What is the __Recall__ Score for this spam predictor? (round to three decimal places)  
Q6. What is the __Accuracy__ of the current model? (round to three decimal places)  

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "7"

In [44]:
# Supply your answers to Q1 through Q8
# in variables a1 through a6.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score

def task_seven():
    a1 = None
    a2 = None
    a3 = None
    a4 = None
    a5 = None
    a6 = None

    # YOUR CODE HERE

    # 1. Get the predictions (matching Task 6 logic)
    lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
    y_pred = lr.predict(X_test)
    
    # 2. Get the confusion matrix components
    # tn: True Neg (Ham correctly identified)
    # fp: False Pos (Ham incorrectly marked as Spam)
    # fn: False Neg (Spam missed)
    # tp: True Pos (Spam correctly identified)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    # Q1: Correctly Predicted spam emails (True Positives)
    a1 = int(tp)
    
    # Q2: False Positive predictions (Type I error)
    a2 = int(fp)
    
    # Q3: False Negative predictions (Type II error)
    a3 = int(fn)
    
    # Q4: Precision Score (TP / (TP + FP))
    a4 = round(float(precision_score(y_test, y_pred)), 3)
    
    # Q5: Recall Score (TP / (TP + FN))
    a5 = round(float(recall_score(y_test, y_pred)), 3)
    
    # Q6: Accuracy (Correct / Total)
    a6 = round(float(accuracy_score(y_test, y_pred)), 3)
  
    #raise NotImplementedError()

    return (a1, a2, a3, a4, a5, a6)

In [45]:
# Autograder tests
print(f"Task {task_id} - AG tests")
stu_ans = task_seven()

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(
    stu_ans, tuple
), f"Task {task_id}: Your function should return a tuple."

assert (
    len(stu_ans) == 6
), f"Task {task_id}: Your answer tuple does not contain the correct number of elements."

assert isinstance(stu_ans[0], int), f"Task {task_id}: Element one should be an integer"
assert isinstance(stu_ans[1], int), f"Task {task_id}: Element two should be an integer"
assert isinstance(
    stu_ans[2], int
), f"Task {task_id}: Element three should be an integer"
assert isinstance(stu_ans[3], float), f"Task {task_id}: Element four should be a float"
assert isinstance(stu_ans[4], float), f"Task {task_id}: Element five should be a float"
assert isinstance(stu_ans[5], float), f"Task {task_id}: Element six should be a float"



del stu_ans

Task 1 - AG tests
Task 1 - your answer:
(392, 43, 68, 0.901, 0.852, 0.904)


<a id='t8'></a>
### Task 8 - Grid Search on Logistic Regression (15 points).

Perform a [GridSearchCV](https://scikit-learn.org/1.2/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV) over the hyper-parameters listed below for a Logistic Regression classifier, optimizing for classifier **precision** for scoring and five-fold cross validation. 

**Note: Use the following parameter settings for the logistic regression:**
 * Use the 'liblinear' solver, which supports both L1 and L2 regularization.
 * Set `random_state=42`, since the solver uses randomization internally.

`'penalty': ['l1', 'l2']`

`'C':[0.005, 0.01, 0.05, 0.1, 1.0, 10.0]`

From `.cv_results_`, create an array of the mean test scores for each hyper-parameter combination. i.e.

|C|L1|L2|  
|:-|-:|:-:|
|__`0.005`__|?|?|  
|**`0.01`**|?|?|  
|**`0.05`**|?|?|  
|**`0.1`**|?|?|  
|**`1.0`**|?|?|  
|**`10.0`**|?|?|  
<br>

*This function should return a 6 by 2 numpy array of floats that contain the values for each "?" above. Do not return a pd.DataFrame.*  


<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "8"

In [54]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
import numpy as np

def task_eight():
    mean_test_scores = None

    # YOUR CODE HERE

    # 1. Setup the parameter grid
    param_grid = {
        'penalty': ['l1', 'l2'],
        'C': [0.005, 0.01, 0.05, 0.1, 1.0, 10.0]
    }
    
    # 2. Initialize the Logistic Regression model
    # liblinear is required for l1 support
    lr = LogisticRegression(solver='liblinear', random_state=42)
    
    # 3. Initialize and fit GridSearchCV
    grid_search = GridSearchCV(lr, param_grid, scoring='precision', cv=5)
    grid_search.fit(X_train, y_train)
    
    # 4. Extract mean test scores
    # cv_results_['mean_test_score'] returns a 1D array of 12 values
    # We reshape it to (6, 2) to match the required output table (C rows, Penalty columns)
    mean_test_scores = grid_search.cv_results_['mean_test_score'].reshape(6, 2)
    
    #raise NotImplementedError()

    return mean_test_scores

In [55]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# task_eight()

In [56]:
# Autograder tests
print(f"Task {task_id} - AG tests")
stu_ans = task_eight()

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(
    stu_ans, np.ndarray
), f"Task {task_id}: Your function should return a np.ndarray. "

assert stu_ans.shape == (
    6,
    2,
), f"Task {task_id}: Your np.ndarray should be of shape (6, 2). "

# Some hidden tests

del stu_ans

Task 1 - AG tests
Task 1 - your answer:
[[0.86896 0.91596]
 [0.90478 0.91791]
 [0.92881 0.92266]
 [0.93051 0.92125]
 [0.92606 0.92687]
 [0.92337 0.92308]]


<a id='heatmapviz'></a>
### GridSearch Heatmap Visualization

<a href='#toc'>TOC</a>

In [57]:
def GridSearch_Heatmap(scores):
    plt.figure(figsize=(8, 6))
    plt.yticks(rotation=0)

    cmap = "plasma"  # "gnuplot" # "viridis" # "magma" #

    sns.heatmap(
        scores.reshape(6, 2),
        xticklabels=["l1", "l2"],
        yticklabels=[0.005, 0.01, 0.05, 0.1, 1, 10],
        cmap=cmap,
    )

    return

In [58]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# GridSearch_Heatmap(task_eight())

<a id='t9'></a>
### Task 9 - Normalizing Features and Regularization (5 points).

Now re-run the code from Question 8 above, but using the raw *unnormalized* training data (i.e. X_train_raw as computed previously as part of Q1.). Return the *highest* precision you obtained from cross-validation in Task 8 using normalized features, and the *highest* precision you obtain with the raw, unnormalized features. 

Your function should return a two-element tuple of floats `(best_precision_normalized, best_precision_unnormalized)`

It is very instructive to compare the results from (a) using correctly normalized features, vs. (b) forgetting to normalize the features and (c) the dummy baseline you computed in Q2.  

<a href='#toc'>TOC</a>

In [59]:
# hidden autograder codeblock
task_id = "9"

In [63]:
def task_nine():
    best_precision_normalized = None
    best_precision_unnormalized = None

    # YOUR CODE HERE
    
    # Define the same grid settings as Task 8
    param_grid = {
        'penalty': ['l1', 'l2'],
        'C': [0.005, 0.01, 0.05, 0.1, 1.0, 10.0]
    }
    
    # INCREASE max_iter to 1000 to help the unnormalized data converge
    lr = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)

    # 1. Grid Search using NORMALIZED features
    grid_norm = GridSearchCV(lr, param_grid, scoring='precision', cv=5)
    grid_norm.fit(X_train, y_train)
    best_precision_normalized = float(grid_norm.best_score_)
    
    # 2. Grid Search using UNNORMALIZED features
    grid_raw = GridSearchCV(lr, param_grid, scoring='precision', cv=5)
    grid_raw.fit(X_train_raw, y_train)
    best_precision_unnormalized = float(grid_raw.best_score_)
    #raise NotImplementedError()

    return (best_precision_normalized, best_precision_unnormalized)

In [64]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# task_nine()

In [ ]:
# Autograder tests
print(f"Task {task_id} - AG tests")
stu_ans = task_nine()

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(
    stu_ans, tuple
), f"Task {task_id}: Your function should return a tuple. "

assert (
    len(stu_ans) == 2
), f"Task {task_id}: The length of your returned tuple should be 2. "

assert all(
    [isinstance(item, float) for item in stu_ans]
), f"Task {task_id}: Your tuple should only contain floats. "

# Some hidden tests

del stu_ans

Task 9 - AG tests


<a href='#toc'>TOC</a>